In [1]:
from bs4 import BeautifulSoup
import requests
import re

In [2]:
url = 'https://www.transfermarkt.pt/lionel-messi/profil/spieler/28003'

player_id = url.split('/')[-1]

In [3]:
headers = {
    'User-agent' : 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36'
}

In [4]:
response = requests.get(url, headers = headers)
response.status_code

200

In [5]:
soup = BeautifulSoup(response.content, 'html.parser')

### Get the name and number with css selector

In [6]:
soup.select_one('h1[class="data-header__headline-wrapper"]') ## css selector

<h1 class="data-header__headline-wrapper">
<span class="data-header__shirt-number">
                        #10                    </span>
                                Lionel <strong>Messi</strong> </h1>

In [7]:
words = soup.select_one('h1[class="data-header__headline-wrapper"]').text.split()
words

['#10', 'Lionel', 'Messi']

In [8]:
name = words[-2] + ' ' + words[-1]
name

'Lionel Messi'

In [9]:
number = int(words[0][1:])
number

10

### Doing the API

The market value history

In [10]:
response = requests.get(f'https://tmapi-alpha.transfermarkt.technology/player/{player_id}/market-value-history', headers=headers)

In [11]:
response.json().keys()

dict_keys(['success', 'message', 'data'])

# Setting the table with players

In [12]:
import numpy as np
import pandas as pd
from urllib.parse import urljoin

In [13]:
season_2025 = pd.DataFrame()

In [14]:
year = '2025'
url = f'https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1/plus/?saison_id={year}'

response = requests.get(url=url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
table_teams = soup.select_one('table[class="items"]')

In [16]:
teams = []

for a in soup.select("table.items tbody td.hauptlink a"):
    name = a.get_text(strip=True)
    href = urljoin(url, a["href"])
    teams.append((name, href))

teams = teams[:-5]

Cleaning the vector

In [18]:
teams = [i for i in teams if i[0] != '']
teams = teams[:20]

In [19]:
teams

[('Real Madrid',
  'https://www.transfermarkt.us/real-madrid/startseite/verein/418/saison_id/2025'),
 ('FC Barcelona',
  'https://www.transfermarkt.us/fc-barcelona/startseite/verein/131/saison_id/2025'),
 ('Atlético de Madrid',
  'https://www.transfermarkt.us/atletico-madrid/startseite/verein/13/saison_id/2025'),
 ('Athletic Bilbao',
  'https://www.transfermarkt.us/athletic-bilbao/startseite/verein/621/saison_id/2025'),
 ('Villarreal CF',
  'https://www.transfermarkt.us/fc-villarreal/startseite/verein/1050/saison_id/2025'),
 ('Real Sociedad',
  'https://www.transfermarkt.us/real-sociedad-san-sebastian/startseite/verein/681/saison_id/2025'),
 ('Real Betis Balompié',
  'https://www.transfermarkt.us/real-betis-sevilla/startseite/verein/150/saison_id/2025'),
 ('Valencia CF',
  'https://www.transfermarkt.us/fc-valencia/startseite/verein/1049/saison_id/2025'),
 ('Girona FC',
  'https://www.transfermarkt.us/fc-girona/startseite/verein/12321/saison_id/2025'),
 ('Celta de Vigo',
  'https://www.

In [20]:
players = []

for i in teams:
    url = i[1]
    response = requests.get(url = url, headers = headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    names = []
    for row in soup.select("table.items tbody tr"):
        cell = row.select_one("td.hauptlink a")
        if not cell:
            continue
        
        name = cell.get_text(strip=True)
        if name not in names:
            link = urljoin(url, cell["href"])
            players.append((i[0], name, link))
            names.append(name)

In [22]:
len(players)

498

In [23]:
for i in players:
    if i[0] == 'Real Madrid': print(i)

('Real Madrid', 'Thibaut Courtois', 'https://www.transfermarkt.us/thibaut-courtois/profil/spieler/108390')
('Real Madrid', 'Andriy Lunin', 'https://www.transfermarkt.us/andriy-lunin/profil/spieler/404839')
('Real Madrid', 'Dean Huijsen', 'https://www.transfermarkt.us/dean-huijsen/profil/spieler/890290')
('Real Madrid', 'Éder Militão', 'https://www.transfermarkt.us/eder-militao/profil/spieler/401530')
('Real Madrid', 'Raúl Asencio', 'https://www.transfermarkt.us/raul-asencio/profil/spieler/935245')
('Real Madrid', 'Antonio Rüdiger', 'https://www.transfermarkt.us/antonio-rudiger/profil/spieler/86202')
('Real Madrid', 'David Alaba', 'https://www.transfermarkt.us/david-alaba/profil/spieler/59016')
('Real Madrid', 'Álvaro Carreras', 'https://www.transfermarkt.us/alvaro-carreras/profil/spieler/811778')
('Real Madrid', 'Fran García', 'https://www.transfermarkt.us/fran-garcia/profil/spieler/341264')
('Real Madrid', 'Ferland Mendy', 'https://www.transfermarkt.us/ferland-mendy/profil/spieler/291

## Generalizing for seasons from 17 to 24, so we get all the data we need

In [24]:
years = [str(2017 + i) for i in range(9)]
players = []


for year in years:
    url = f'https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1/plus/?saison_id={year}'
    response = requests.get(url=url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    table_teams = soup.select_one('table[class="items"]')  # table of teams per season

    teams = []

    for a in soup.select("table.items tbody td.hauptlink a"):
        name = a.get_text(strip=True)
        href = urljoin(url, a["href"])
        teams.append((name, href))
    
    teams = teams[:-5]  
    
    teams = [i for i in teams if i[0] != '']
    teams = teams[:20]  # here a vector of all teams cleaned
    
    for i in teams:
        url = i[1]
        response = requests.get(url = url, headers = headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        names = []
        for row in soup.select("table.items tbody tr"):
            cell = row.select_one("td.hauptlink a")
            if not cell:
                continue
            
            name = cell.get_text(strip=True)
            if name not in names:
                link = urljoin(url, cell["href"])
                player_id = link.split('/')[-1]
                players.append((year, i[0], player_id, name, link))
                names.append(name)

In [25]:
len(players)

6384

In [26]:
players[:5]

[('2017',
  'FC Barcelona',
  '74857',
  'Marc-André ter Stegen',
  'https://www.transfermarkt.us/marc-andre-ter-stegen/profil/spieler/74857'),
 ('2017',
  'FC Barcelona',
  '146227',
  'Jasper Cillessen',
  'https://www.transfermarkt.us/jasper-cillessen/profil/spieler/146227'),
 ('2017',
  'FC Barcelona',
  '142033',
  'Adrián Ortolá',
  'https://www.transfermarkt.us/adrian-ortola/profil/spieler/142033'),
 ('2017',
  'FC Barcelona',
  '126540',
  'Samuel Umtiti',
  'https://www.transfermarkt.us/samuel-umtiti/profil/spieler/126540'),
 ('2017',
  'FC Barcelona',
  '18944',
  'Gerard Piqué',
  'https://www.transfermarkt.us/gerard-pique/profil/spieler/18944')]

In [27]:
columns = ['year', 'club', 'player_id', 'player_name', 'link_html']
df = pd.DataFrame(players, columns = columns)
df

,year,club,player_id,player_name,link_html
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...
...,...,...,...,...,...
6379,2025,Real Oviedo,337284,Ovie Ejaria,https://www.transfermarkt.us/ovie-ejaria/profi...
6380,2025,Real Oviedo,620164,Haissem Hassan,https://www.transfermarkt.us/haissem-hassan/pr...
6381,2025,Real Oviedo,578419,Federico Viñas,https://www.transfermarkt.us/federico-vinas/pr...
6382,2025,Real Oviedo,743951,Thiago Borbas,https://www.transfermarkt.us/thiago-borbas/pro...


In [30]:
from pathlib import Path

base_path = Path("/Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value")
base_path.mkdir(parents=True, exist_ok=True)

df.to_parquet(base_path / "players_panel.parquet", index=False)

In [32]:
df = pd.read_parquet(base_path / "players_panel.parquet")
df.head()

,year,club,player_id,player_name,link_html
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...


In [96]:
def value_near_december(df, year):
    dec = df[(df.date.dt.year == year) & (df.date.dt.month == 12)]
    if not dec.empty:
        return dec.sort_values('date').iloc[-1]

    nov = df[(df.date.dt.year == year) & (df.date.dt.month == 11)]
    if not nov.empty:
        return nov.sort_values('date').iloc[-1]

    jan = df[(df.date.dt.year == year + 1) & (df.date.dt.month == 1)]
    if not jan.empty:
        return jan.sort_values('date').iloc[0]

    return None

In [110]:
# for each player:
# get the age, the value on the middle of the season, 
# its nationality, its height, its foot, its position, how long he still has of contract

# first for 1 player

url = df.iloc[0]['link_html']
player_id = df.iloc[0]['player_id']
response = requests.get(url=url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')


year = int(df.iloc[0]['year'])

# age
age = soup.select_one('span[itemprop="birthDate"]')
age = year - int(str(span).split()[-3])
age

# nationality
nationality = soup.select_one('span[itemprop="nationality"]').get_text(strip=True)
nationality

# height
height = soup.select_one('span[itemprop="height"]').get_text(strip=True)[:4]
height = float(height.replace(',','.'))
height

# prefered foot
foot = next(
    label.find_next_sibling('span', class_='info-table__content--bold').get_text(strip=True)
    for label in soup.select('span.info-table__content--regular')
    if label.get_text(strip=True) == 'Foot:'
)
foot

# position
position = next(
    label.find_next_sibling('span', class_='info-table__content--bold').get_text(strip=True)
    for label in soup.select('span.info-table__content--regular')
    if label.get_text(strip=True) == 'Position:'
)
position

# market value
url = url.replace('profil', 'marktwertverlauf')
response = requests.get(url=url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
soup

api_link = f'https://tmapi-alpha.transfermarkt.technology/player/{player_id}/market-value-history'

response = requests.get(url=api_link, headers=headers)

json_root = response.json()      
data = json_root.get('data', {})
history = data.get('history', [])
current = data.get('current', {})

df_history = pd.DataFrame([
    {
        'player_id': h['playerId'],
        'club_id': h.get('clubId'),
        'date': pd.to_datetime(h['marketValue']['determined']),
        'age': h.get('age'),
        'market_value_eur': h['marketValue']['value']
    }
    for h in history
])

market_value = value_near_december(df_history, year)
market_value = market_value['market_value_eur']
market_value

np.int64(40000000)

In [111]:
vector_age = []

for i in range(len(df)):
    url = df.iloc[i]['link_html']
    player_id = df.iloc[i]['player_id']
    response = requests.get(url=url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    year = int(df.iloc[i]['year'])
    
    # age
    
    age = soup.select_one('span[itemprop="birthDate"]')
    age = year - int(str(span).split()[-3])
    vector_age.append(age)

print(vector_age)

KeyboardInterrupt: 

In [ ]:
df['age'] = vector_age

In [ ]:
df.to_parquet(base_path / "players_panel.parquet", index=False)